# Visual test: **ArcFace fine-tuned checkpoint**

Loads **`finetune_arcface_best.pt`** from the project root (produced by `fine_tune_arcface.ipynb`) and shows **image pairs** with **cosine similarity** — no training.

Set **`MERGE_BOTH_GALLERIES`** the same as when you trained so file lists line up with expectations. Embeddings use **`labels=None`** (backbone + FC only); class IDs in the checkpoint are not needed for similarity plots.


## 0) Imports & device

In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)

## 1) Checkpoint + gallery paths

Change **`CKPT_PATH`** if your weights live elsewhere.

In [ ]:
PROJECT_ROOT = Path.cwd()
CKPT_PATH = PROJECT_ROOT / "finetune_arcface_best.pt"
if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Missing {CKPT_PATH}. Run fine_tune_arcface.ipynb first or set CKPT_PATH.")

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
label_to_idx = ckpt["label_to_idx"]
identities_ckpt = ckpt["identities"]
num_classes = len(identities_ckpt)
EMBEDDING_DIM = int(ckpt["embedding_dim"])
IMG_SIZE = int(ckpt.get("img_size", 112))
print("Checkpoint epoch:", ckpt.get("epoch", "?"))
print("Classes:", num_classes, "| img_size:", IMG_SIZE, "| emb_dim:", EMBEDDING_DIM)
print("train_gallery:", ckpt.get("train_gallery", "?"), "| merged:", ckpt.get("merge_galleries", "?"))

DATASET_ROOT = PROJECT_ROOT / "open_data_set"
PHOTOS_ALL_FACES = DATASET_ROOT / "photos_all_faces"
PHOTOS_CROPPED = DATASET_ROOT / "photos_all_faces_cropped"
MERGE_BOTH_GALLERIES = False


def list_images(folder: Path):
    out = []
    if not folder.exists():
        return out
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        out.extend(folder.glob(ext))
    return sorted(out)


def infer_identity(path: Path) -> str:
    return path.stem.split("_")[0].upper()


rows = []
seen_paths = set()
if MERGE_BOTH_GALLERIES:
    for folder in (PHOTOS_CROPPED, PHOTOS_ALL_FACES):
        for p in list_images(folder):
            key = str(p.resolve())
            if key in seen_paths:
                continue
            seen_paths.add(key)
            rows.append({"path": key, "identity": infer_identity(p)})
    print("Gallery: merged cropped + photos_all_faces")
else:
    if PHOTOS_CROPPED.is_dir() and list_images(PHOTOS_CROPPED):
        gdir = PHOTOS_CROPPED
    elif PHOTOS_ALL_FACES.is_dir() and list_images(PHOTOS_ALL_FACES):
        gdir = PHOTOS_ALL_FACES
    else:
        raise FileNotFoundError(f"No images under {PHOTOS_CROPPED} or {PHOTOS_ALL_FACES}")
    print("Gallery:", gdir.resolve())
    for p in list_images(gdir):
        rows.append({"path": str(p.resolve()), "identity": infer_identity(p)})

gallery_df_all = pd.DataFrame(rows)
cnt = gallery_df_all["identity"].value_counts()
gallery_df = gallery_df_all[gallery_df_all["identity"].isin(cnt[cnt >= 2].index)].reset_index(drop=True)
print(f"Images for pairing: {len(gallery_df)} | IDs with 2+ images: {gallery_df['identity'].nunique()}")
print(f"Full gallery (for MTCNN matching): {len(gallery_df_all)} images | {gallery_df_all['identity'].nunique()} IDs")

ids_in_ckpt = set(identities_ckpt)
ids_data = set(gallery_df["identity"].unique())
only_data = ids_data - ids_in_ckpt
only_ckpt = ids_in_ckpt - ids_data
if only_data or only_ckpt:
    print("Note: identity mismatch vs checkpoint (plots still work):")
    if only_data:
        print("  in data only:", sorted(only_data)[:20], "..." if len(only_data) > 20 else "")
    if only_ckpt:
        print("  in ckpt only:", sorted(only_ckpt)[:20], "..." if len(only_ckpt) > 20 else "")

## 2) Model (must match `fine_tune_arcface.ipynb`)

In [ ]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features: int, out_features: int, s: float = 30.0, m: float = 0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        emb = F.normalize(emb, dim=1)
        W = F.normalize(self.weight, dim=1)
        cosine = F.linear(emb, W)
        sine = torch.sqrt(torch.clamp(1.0 - cosine.pow(2), min=0.0, max=1.0))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s


class FaceNet(nn.Module):
    def __init__(self, embedding_dim: int = 512, num_classes: int = 11):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.bn = nn.BatchNorm1d(2048)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(2048, embedding_dim)
        self.arc = ArcMarginProduct(embedding_dim, num_classes, s=30.0, m=0.35)

    def forward(self, x: torch.Tensor, labels=None):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.bn(x)
        x = self.dropout(x)
        emb = self.fc(x)
        if labels is None:
            return F.normalize(emb, dim=1)
        return self.arc(emb, labels)


model = FaceNet(embedding_dim=EMBEDDING_DIM, num_classes=num_classes).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print("Loaded weights OK.")

## 3) Preprocess + embeddings (eval / no augment)

In [ ]:
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def embed_path(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    x = val_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        e = model(x, labels=None).cpu().numpy()[0]
    return e


def cos_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def show_pair(path_a: str, path_b: str, title: str, figsize=(6, 3)):
    ia = Image.open(path_a).convert("RGB")
    ib = Image.open(path_b).convert("RGB")
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    ax[0].imshow(ia)
    ax[0].axis("off")
    ax[0].set_title(Path(path_a).name, fontsize=8)
    ax[1].imshow(ib)
    ax[1].axis("off")
    ax[1].set_title(Path(path_b).name, fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 4) Visual test cases

Adjust **`N_SAME_ROWS`** / **`N_DIFF_ROWS`** (how many figure rows to show).

In [ ]:
%matplotlib inline
N_SAME_ROWS = 4
N_DIFF_ROWS = 4
rng = np.random.default_rng(SEED)

uids = sorted(gallery_df["identity"].unique())
if len(uids) == 0:
    raise RuntimeError("No identities with 2+ images.")

print("=== Same person (two different images) ===")
pick_same = list(uids)
rng.shuffle(pick_same)
for uid in pick_same[:N_SAME_ROWS]:
    sub = gallery_df[gallery_df["identity"] == uid].sort_values("path")
    p0, p1 = sub.iloc[0]["path"], sub.iloc[1]["path"]
    s = cos_sim(embed_path(p0), embed_path(p1))
    show_pair(p0, p1, f"SAME ID {uid}  |  cos = {s:.4f}")

if len(uids) >= 2:
    print("=== Different people (random pairs) ===")
    for _ in range(N_DIFF_ROWS):
        a, b = rng.choice(uids, size=2, replace=False)
        pa = rng.choice(gallery_df[gallery_df["identity"] == a]["path"].tolist())
        pb = rng.choice(gallery_df[gallery_df["identity"] == b]["path"].tolist())
        s = cos_sim(embed_path(pa), embed_path(pb))
        show_pair(pa, pb, f"DIFF  {a} vs {b}  |  cos = {s:.4f}")
else:
    print("Skip diff pairs: need 2+ identities.")

## 5) Optional: numeric summary (no plots)

Quick mean cosine for same-ID vs diff-ID over random samples.

In [ ]:
N_SAME_STATS = min(50, len(uids))
N_DIFF_STATS = 80

same_sims = []
for uid in rng.choice(uids, size=min(N_SAME_STATS, len(uids)), replace=False):
    sub = gallery_df[gallery_df["identity"] == uid]
    if len(sub) < 2:
        continue
    idx = rng.choice(len(sub), size=2, replace=False)
    p0, p1 = sub.iloc[idx[0]]["path"], sub.iloc[idx[1]]["path"]
    same_sims.append(cos_sim(embed_path(p0), embed_path(p1)))

diff_sims = []
if len(uids) >= 2:
    for _ in range(N_DIFF_STATS):
        a, b = rng.choice(uids, size=2, replace=False)
        pa = rng.choice(gallery_df[gallery_df["identity"] == a]["path"].tolist())
        pb = rng.choice(gallery_df[gallery_df["identity"] == b]["path"].tolist())
        diff_sims.append(cos_sim(embed_path(pa), embed_path(pb)))

if same_sims:
    print(f"Same-ID (n={len(same_sims)}): mean={np.mean(same_sims):.4f}  min={np.min(same_sims):.4f}  max={np.max(same_sims):.4f}")
if diff_sims:
    print(f"Diff-ID (n={len(diff_sims)}): mean={np.mean(diff_sims):.4f}  min={np.min(diff_sims):.4f}  max={np.max(diff_sims):.4f}")

## 6) MTCNN detection + your model — **`cd_gp_0_eo_12.JPG`**

Same visual test case as **`people_identification_in_the_wild_Aggreggate.ipynb` §5.1**: scene `open_data_set/photos_all/cd_gp_0_eo_12.JPG`, filename token **CD** → expected subjects **C** and **D** (four faces may be detected).

Uses **DeepFace `extract_faces`** with **`detector_backend="mtcnn"`** (aligns crops). Each crop is embedded with **this notebook’s fine-tuned `FaceNet`**. Matching follows the aggregate idea: **mean cosine similarity** per gallery identity, **argmax**, optional **accept** if best mean ≥ **`MATCH_MIN_MEAN_COS`** (tune like a threshold).

Requires **`deepface`** in the venv (`pip install -r requirements.txt`).

In [ ]:
import cv2
from deepface import DeepFace

TEST_SCENE = DATASET_ROOT / "photos_all" / "cd_gp_0_eo_12.JPG"
MATCH_THRESHOLD = 0.05  # Aggregate §6.5: accept if best mean |1-cos| <= this


def cosine_dist_abs(q: np.ndarray, g: np.ndarray) -> float:
    return abs(1.0 - cos_sim(q, g))


scene_bgr = cv2.imread(str(TEST_SCENE))
if scene_bgr is None:
    raise FileNotFoundError(f"Cannot read {TEST_SCENE} (same path as Aggregate notebook §5.1).")

_h0, _w0 = scene_bgr.shape[:2]
try:
    _raw_faces = DeepFace.extract_faces(
        img_path=scene_bgr,
        detector_backend="mtcnn",
        enforce_detection=True,
        align=True,
    )
except Exception:
    _raw_faces = []

test_boxes = []
aligned_faces = []
for _item in _raw_faces:
    _a = _item["facial_area"]
    _x = max(0, min(int(_a["x"]), _w0 - 1))
    _y = max(0, min(int(_a["y"]), _h0 - 1))
    _w = max(1, min(int(_a["w"]), _w0 - _x))
    _h = max(1, min(int(_a["h"]), _h0 - _y))
    if _w < 10 or _h < 10:
        continue
    test_boxes.append({"x": _x, "y": _y, "w": _w, "h": _h})
    face_arr = _item.get("face")
    if face_arr is not None:
        fa = np.asarray(face_arr)
        if fa.dtype != np.uint8:
            fa = np.clip(fa * 255.0, 0, 255).astype(np.uint8)
        if fa.ndim == 3 and fa.shape[2] >= 3:
            aligned_faces.append(fa[:, :, :3])
        else:
            aligned_faces.append(None)
    else:
        aligned_faces.append(None)

tok = Path(TEST_SCENE).stem.split("_")[0].upper()
expected_ids = set(tok) if tok.isalpha() else set()
print(f"Scene: {TEST_SCENE.name} | {_w0}x{_h0} | MTCNN detections: {len(test_boxes)}")
print("Expected from filename token:", sorted(expected_ids))

_vis_bgr = scene_bgr.copy()
for _i, _b in enumerate(test_boxes):
    cv2.rectangle(
        _vis_bgr,
        (_b["x"], _b["y"]),
        (_b["x"] + _b["w"], _b["y"] + _b["h"]),
        (0, 255, 0),
        2,
    )
    cv2.putText(
        _vis_bgr,
        str(_i),
        (_b["x"] + 2, _b["y"] + 14),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 255, 0),
        1,
    )
_vis = cv2.cvtColor(_vis_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 7))
plt.imshow(_vis)
plt.title(
    f"MTCNN ({len(test_boxes)}) — {TEST_SCENE.name} | expected: {', '.join(sorted(expected_ids))}"
)
plt.axis("off")
plt.show()


def embed_rgb_uint8(rgb: np.ndarray) -> np.ndarray:
    img = Image.fromarray(rgb)
    x = val_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        e = model(x, labels=None).cpu().numpy()[0]
    return e


gallery_vecs_by_id = {lab: [] for lab in identities_ckpt}
gallery_paths_by_id = {lab: [] for lab in identities_ckpt}
for _, row in gallery_df_all.iterrows():
    lab = row["identity"]
    if lab not in gallery_vecs_by_id:
        continue
    gallery_vecs_by_id[lab].append(embed_path(row["path"]))
    gallery_paths_by_id[lab].append(row["path"])

labels_rank = sorted(lab for lab in identities_ckpt if gallery_vecs_by_id[lab])
print("Gallery IDs with images:", labels_rank)


def match_aggregate_mean(q: np.ndarray):
    if not labels_rank:
        return None, float("nan"), False, {}, None
    per_id_mean = {}
    for lab in labels_rank:
        vecs = gallery_vecs_by_id[lab]
        if not vecs:
            continue
        per_id_mean[lab] = float(np.mean([cosine_dist_abs(q, v) for v in vecs]))
    if not per_id_mean:
        return None, float("nan"), False, {}, None
    best_lab = min(per_id_mean, key=per_id_mean.get)
    best_mean = per_id_mean[best_lab]
    vecs = gallery_vecs_by_id[best_lab]
    paths = gallery_paths_by_id[best_lab]
    di = int(np.argmin([cosine_dist_abs(q, v) for v in vecs]))
    best_path = paths[di]
    accepted = best_mean <= MATCH_THRESHOLD
    return best_lab, best_mean, accepted, per_id_mean, best_path


scene_rgb = cv2.cvtColor(scene_bgr, cv2.COLOR_BGR2RGB)
_n = len(test_boxes)
if _n == 0:
    print("No faces detected.")
else:
    fig, axes = plt.subplots(_n, 2, figsize=(6, 3 * _n))
    if _n == 1:
        axes = [axes]

    for _i, _b in enumerate(test_boxes):
        if _i < len(aligned_faces) and aligned_faces[_i] is not None:
            crop_rgb = aligned_faces[_i]
            _left_title = f"Scene face #{_i} (MTCNN aligned)"
        else:
            crop_bgr = scene_bgr[_b["y"] : _b["y"] + _b["h"], _b["x"] : _b["x"] + _b["w"]]
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
            _left_title = f"Scene face #{_i} (bbox crop)"

        qe = embed_rgb_uint8(crop_rgb)
        pred_id, mean_dist, accepted, per_id_mean, gallery_path = match_aggregate_mean(qe)
        match_label = "MATCH" if accepted else "NO MATCH"
        color = "green" if accepted else "red"

        axes[_i][0].imshow(crop_rgb)
        axes[_i][0].set_title(_left_title, fontsize=10)
        axes[_i][0].axis("off")

        if gallery_path is None:
            axes[_i][1].text(0.5, 0.5, "No gallery", ha="center")
            axes[_i][1].axis("off")
        else:
            g_bgr = cv2.imread(gallery_path)
            if g_bgr is None:
                axes[_i][1].text(0.5, 0.5, "Bad path", ha="center")
            else:
                g_rgb = cv2.cvtColor(g_bgr, cv2.COLOR_BGR2RGB)
                axes[_i][1].imshow(g_rgb)
            axes[_i][1].set_title(
                f"→ {pred_id} (mean_dist={mean_dist:.4f}) [{match_label}]",
                fontsize=10,
                color=color,
            )
            axes[_i][1].axis("off")

        top3 = sorted(per_id_mean.items(), key=lambda t: t[1])[:3]
        print(
            f"Face #{_i}: pred={pred_id}  mean_dist={mean_dist:.4f}  {match_label}  |  top3(mean dist): {top3}"
        )

    fig.suptitle(
        f"MTCNN + fine-tuned model — {TEST_SCENE.name}\n"
        f"Expected: {', '.join(sorted(expected_ids))}  |  MATCH if mean |1-cos| ≤ {MATCH_THRESHOLD}",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()